# ReVerb — verb-conditioned 3D affordance prediction (demo)

**ReVerb** predicts, for a given **verb**, *where on a 3D object that action applies* — a per-vertex
affordance map on geometry reconstructed from a single image. This notebook loads a trained model and a
pre-reconstructed object, and shows the core behaviour: **the same object routes to different regions for
different verbs**.

Full pipeline: `single RGB image → SAM3D reconstruction → per-vertex DINOv2 + geometry → verb-conditioned GNN`.
Here we start from a reconstructed object and run the prediction step.

In [ ]:
import sys
sys.path.insert(0, "src")            # run this notebook from the repo root
import numpy as np, torch
import matplotlib.pyplot as plt

# --- environment-specific paths (point these at your own data / checkpoint) ---
RECON_DIR = "/home/datasets/customDatasets/cmr2/reconstructions"
CKPT      = "outputs/gnn_lean8_fold0.pt"   # one trained ReVerb fold
DEVICE    = "cpu"

## 1 · Load the trained model and the verb encoder

In [ ]:
from models.gnn_head import AffordanceGNN, AffordanceGNNConfig
from vlm.vlm_wrapper import VLMWrapper, VLMConfig

ckpt  = torch.load(CKPT, map_location=DEVICE, weights_only=False)
cfg   = AffordanceGNNConfig(**ckpt["cfg"])
model = AffordanceGNN(cfg).to(DEVICE); model.load_state_dict(ckpt["model"]); model.eval()

verb_encoder = VLMWrapper(VLMConfig(device=DEVICE))   # CLIP text tower encodes the verb
print(f"ReVerb loaded — verb-conditioned spatial GNN (k={cfg.knn_k}, {cfg.gnn_layers} layers)")

## 2 · Load a reconstructed object

Each vertex carries the two learned/handcrafted channels of the lean model: **DINOv2 appearance (128-d)** and
**local geometry (5-d)**. (CLIP-vision, SLAT and normals were ablated to ≈0 and dropped.)

In [ ]:
OBJ = "cup__14_167_1083__v000"          # a reconstructed cup with contain / pour / grasp regions
d   = f"{RECON_DIR}/{OBJ}"
pos  = torch.load(f"{d}/vertex_positions.pt", weights_only=False).float()             # (V, 3)
dino = torch.load(f"{d}/vertex_dino_fine.pt", weights_only=False)["features"].float() # (V, 128)
geom = torch.load(f"{d}/vertex_geom.pt",      weights_only=False).float()             # (V, 5)
print(f"{OBJ}: {len(pos):,} vertices")

# subsample for a fast interactive demo (prediction is per-vertex; any subset works)
rng = np.random.default_rng(0)
sel = np.sort(rng.choice(len(pos), min(20000, len(pos)), replace=False))
idx = torch.as_tensor(sel)
pos_s, dino_s, geom_s = pos[idx], dino[idx], geom[idx]
knn = AffordanceGNN._build_knn(pos_s, cfg.knn_k)   # kNN graph over the sampled vertices

## 3 · Predict a verb-conditioned affordance map

In [ ]:
def predict(verb: str) -> np.ndarray:
    """Per-vertex affordance probability for `verb`. Lean model: feed DINO + geometry + verb-text;
    zero the dropped channels (CLIP-vision / SLAT / normals)."""
    e = verb_encoder.encode_text([verb])[0].to(DEVICE)
    z = lambda dim: torch.zeros(len(sel), dim)
    with torch.no_grad():
        logits = model(e, vlm_features=z(cfg.vlm_dim), dino_vertex=dino_s,
                       slat_vertex=z(cfg.sam3d_dim), vertex_normals=z(cfg.normals_dim),
                       vertex_geom=geom_s, knn_idx=knn)
    return torch.sigmoid(logits).cpu().numpy().reshape(-1)

aff = predict("pour")
print("pour → mean %.3f, active fraction %.3f" % (aff.mean(), (aff > 0.5).mean()))

## 4 · Visualise the map on the object

In [ ]:
def show(verb, ax=None):
    a = predict(verb)
    ax = ax or plt.figure(figsize=(4, 4)).add_subplot(111, projection="3d")
    P = pos_s.numpy()
    ax.scatter(P[:, 0], P[:, 2], P[:, 1], c=a, cmap="turbo", s=2,
               vmin=0, vmax=max(0.5, np.percentile(a, 99.5)))
    ax.set_title(f'"{verb}"'); ax.set_axis_off(); ax.view_init(elev=20, azim=-70)
    return ax

show("pour"); plt.show()

## 5 · Verb routing — the core behaviour

The same object, conditioned on different verbs, lights up **different, near-disjoint regions**: *contain* fills
the interior, *pour* the rim, *grasp* the side. The verb is not redundant with the object — it selects the region.

In [ ]:
verbs = ["contain", "pour", "grasp"]
fig = plt.figure(figsize=(4 * len(verbs), 4))
for i, v in enumerate(verbs):
    show(v, fig.add_subplot(1, len(verbs), i + 1, projection="3d"))
fig.suptitle("Same object, different verb → different region  (verb routing)")
plt.tight_layout(); plt.show()

### Notes

- **Paths are environment-specific** — point `RECON_DIR` / `CKPT` at your own reconstructions and a trained fold.
- To predict on a **new image**, first reconstruct it (`scripts/generate_sam3d.py`) and compute features
  (`scripts/generate_vertex_dino.py`, `scripts/precompute_geom.py`), then run the cells above.
- Method, results and analyses: `docs/RESULTS_consolidated.md`, `docs/OVERNIGHT_FINDINGS.md`.